<a href="https://colab.research.google.com/github/Anish-000/Quantum-Walk-Edge-Detection/blob/main/Quantum_Walk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Numerical Implementation***

In [ ]:
# CELL 0
from google.colab import files
uploaded = files.upload()
# upload both 29030.jpg AND 29030.mat when the picker opens
# you can select multiple files at once

In [ ]:
# ============================================
# CELL 1 — AUTO-DETECT AND VISUALIZE ALL
# BSDS500 IMAGES WITH HUMAN GROUND TRUTH
# Works with any number of image/mat pairs
# ============================================

import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import glob

# ---- CLEAR PREVIOUS DATA ----
all_images        = {}
all_ground_truths = {}

# ---- AUTO-DETECT ALL UPLOADED IMAGE/MAT PAIRS ----
content_dir = '/content'

# find all jpg files
jpg_files = sorted(glob.glob(os.path.join(content_dir, '*.jpg')))

# find matching mat files
image_pairs = []
for jpg_path in jpg_files:
    base_name = os.path.splitext(os.path.basename(jpg_path))[0]
    mat_path  = os.path.join(content_dir, base_name + '.mat')
    if os.path.exists(mat_path):
        image_pairs.append((base_name, jpg_path, mat_path))
    else:
        print(f"Warning: no MAT file found for {base_name}.jpg — skipping")

print(f"Found {len(image_pairs)} valid image/annotation pairs")
print(f"Images: {[p[0] for p in image_pairs]}")
print()

# ---- STORAGE FOR ALL GROUND TRUTHS ----
# dictionary keyed by image name for use in later cells
all_ground_truths = {}
all_images        = {}

# ---- PROCESS EACH IMAGE ----
for idx, (name, jpg_path, mat_path) in enumerate(image_pairs):

    print(f"{'='*55}")
    print(f"Image {idx+1}/{len(image_pairs)}: {name}.jpg")
    print(f"{'='*55}")

    # load image
    img   = Image.open(jpg_path).convert('L')
    image = np.array(img, dtype=float)
    print(f"Image size : {image.shape}")

    # load mat file
    mat              = sio.loadmat(mat_path)
    gt               = mat['groundTruth']
    num_annotators   = gt.shape[1]
    print(f"Annotators : {num_annotators}")

    # extract all annotations
    all_boundaries = []
    for i in range(num_annotators):
        boundary = gt[0, i]['Boundaries'][0, 0]
        all_boundaries.append(boundary)
        print(f"  Annotator {i+1}: {boundary.sum()} edge pixels "
              f"({100*boundary.sum()/boundary.size:.1f}%)")

    # consensus ground truth
    consensus = np.any(all_boundaries, axis=0).astype(int)
    print(f"  Consensus : {consensus.sum()} edge pixels "
          f"({100*consensus.sum()/consensus.size:.1f}%)")

    # check size match
    if consensus.shape != image.shape:
        print(f"  Warning: size mismatch — "
              f"image {image.shape} vs GT {consensus.shape}")
        print(f"  Resizing ground truth to match image...")
        from PIL import Image as PILImage
        gt_img    = PILImage.fromarray(consensus.astype(np.uint8))
        gt_img    = gt_img.resize(
                        (image.shape[1], image.shape[0]),
                        PILImage.NEAREST)
        consensus = np.array(gt_img)

    # store for later cells
    all_ground_truths[name] = consensus
    all_images[name]        = image
    print()

    # visualize
    ncols = num_annotators + 2
    fig, axes = plt.subplots(
        1, ncols,
        figsize=(3.5 * ncols, 3.5))
    fig.suptitle(
        f'BSDS500 Human Annotations — {name}.jpg '
        f'({image.shape[0]}×{image.shape[1]})',
        fontsize=11)

    # original image
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original', fontsize=9)
    axes[0].axis('off')

    # each annotator
    for i in range(num_annotators):
        axes[i+1].imshow(all_boundaries[i], cmap='gray')
        axes[i+1].set_title(
            f'Annotator {i+1}\n{all_boundaries[i].sum()} edges',
            fontsize=8)
        axes[i+1].axis('off')

    # consensus
    axes[-1].imshow(consensus, cmap='gray')
    axes[-1].set_title(
        f'Consensus\n{consensus.sum()} edges',
        fontsize=9)
    axes[-1].axis('off')

    plt.tight_layout()
    plt.show()

# ---- SUMMARY ----
print(f"{'='*55}")
print(f"SUMMARY")
print(f"{'='*55}")
print(f"Total images loaded       : {len(all_ground_truths)}")
print(f"Images ready for testing  : {list(all_ground_truths.keys())}")
print()
avg_edge_density = np.mean([
    100 * gt.sum() / gt.size
    for gt in all_ground_truths.values()
])
print(f"Average edge density      : {avg_edge_density:.2f}%")
print()
print("All ground truths stored in: all_ground_truths dictionary")
print("All images stored in      : all_images dictionary")
print()
print("Cell 1 done. Run Cell 2 for batch algorithm evaluation.")

In [ ]:
# ============================================
# CELL 2 — PAPER'S ALGORITHM + FULL EVALUATION
#
# Contains:
#   PART 1 — DTQWS reproduction (Eq.1–6)
#   PART 2 — Raw oracle evaluation (Eq.3 mask
#             directly vs BSDS500 ground truth)
#   PART 3 — Otsu Oracle F1 (Otsu on gradient,
#             pre-quantum-walk)
#   PART 4 — Otsu Quantum F1 (Otsu on prob_map,
#             post-quantum-walk)
#
# Reads  : all_images, all_ground_truths (Cell 1)
# Outputs: paper_results, paper_raw_oracle,
#          paper_otsu_oracle, paper_otsu_quantum
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import time
from skimage.filters import threshold_otsu

# --------------------------------------------
# PARAMETERS
# --------------------------------------------
s         = 0.0001   # self-loop weight
threshold = 30       # gradient threshold a_th (Eq.3)
max_iter  = 1000     # max walk iterations

# --------------------------------------------
# COIN BASIS
# --------------------------------------------
R, L, U, D, SLF = 0, 1, 2, 3, 4


# ════════════════════════════════════════════
# SHARED HELPERS
# ════════════════════════════════════════════

# --------------------------------------------
# EQ.(2) — initial coin state
# --------------------------------------------
def make_coin_vector(s):
    cv = np.array(
        [1, 1, 1, 1, np.sqrt(s)],
        dtype=np.complex128
    )
    cv /= np.linalg.norm(cv)
    return cv

# --------------------------------------------
# EQ.(3) — gradient (4-direction, paper-exact)
# --------------------------------------------
def compute_gradient(image):

    rows, cols = image.shape
    grad = np.zeros((rows, cols))

    for r in range(rows):
        for c in range(cols):

            center = image[r, c]
            vals   = []

            if c < cols - 1:
                vals.append(abs(center - image[r, c+1]))
            if c > 0:
                vals.append(abs(center - image[r, c-1]))
            if r < rows - 1:
                vals.append(abs(center - image[r+1, c]))
            if r > 0:
                vals.append(abs(center - image[r-1, c]))

            grad[r, c] = max(vals) if vals else 0

    return grad

# --------------------------------------------
# EQ.(4) — modified CG coin operator
# --------------------------------------------
def apply_CG(state, edge_mask, cv):

    out = state.copy()

    # oracle: flip phase of self-loop at marked vertices
    out[edge_mask, SLF] *= -1

    # Grover diffusion
    proj = np.einsum('xyc,c->xy', out, cv.conj())
    out  = (
        2.0 * proj[:, :, None] * cv[None, None, :]
        - out
    )

    return out

# --------------------------------------------
# EQ.(5) — flip-flop shift operator
# --------------------------------------------
def apply_shift(state):

    new = np.zeros_like(state)

    new[:, :, L]   += np.roll(state[:, :, R],  1, axis=1)
    new[:, :, R]   += np.roll(state[:, :, L], -1, axis=1)
    new[:, :, D]   += np.roll(state[:, :, U],  1, axis=0)
    new[:, :, U]   += np.roll(state[:, :, D], -1, axis=0)
    new[:, :, SLF] += state[:, :, SLF]

    return new

# --------------------------------------------
# METRICS
# --------------------------------------------
def compute_metrics(pred, gt):

    TP = np.sum((pred == 1) & (gt == 1))
    FP = np.sum((pred == 1) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    return precision, recall, f1

# --------------------------------------------
# OTSU BINARIZATION
# --------------------------------------------
def otsu_edge_map(signal):
    try:
        t = threshold_otsu(signal)
    except ValueError:
        t = signal.max()
    return (signal >= t).astype(np.uint8)

# --------------------------------------------
# MAIN — run DTQWS on one image
# --------------------------------------------
def run_dtqws(image, s=0.0001, threshold=30, max_iter=1000):

    rows, cols = image.shape
    N          = rows * cols

    grad      = compute_gradient(image)
    edge_mask = grad >= threshold    # T_M from Eq.(3)
    M         = int(edge_mask.sum())

    cv = make_coin_vector(s)

    # Eq.(1),(2) — initial state
    coin_amps = np.array(
        [1, 1, 1, 1, np.sqrt(s)],
        dtype=np.complex128
    ) / np.sqrt(4 + s)

    state = np.zeros((rows, cols, 5), dtype=np.complex128)
    for d in range(5):
        state[:, :, d] = coin_amps[d] / np.sqrt(N)

    ps_history = []
    best_ps    = -1
    best_t     = 0
    best_state = state.copy()

    for t in range(1, max_iter + 1):

        state = apply_CG(state, edge_mask, cv)
        state = apply_shift(state)

        # Eq.(6) — success probability
        ps = float(np.sum(np.abs(state[edge_mask])**2))
        ps_history.append(ps)

        if ps > best_ps:
            best_ps    = ps
            best_t     = t
            best_state = state.copy()

    prob_map = np.sum(np.abs(best_state)**2, axis=2)

    return {
        "grad":       grad,
        "edge_mask":  edge_mask,
        "prob_map":   prob_map,
        "ps_history": ps_history,
        "best_ps":    best_ps,
        "best_t":     best_t,
        "M":          M,
        "N":          N
    }


# ════════════════════════════════════════════
# PART 1 — DTQWS BATCH RUN
# ════════════════════════════════════════════

paper_results = {}

print("=" * 70)
print("CELL 2 — PAPER'S ALGORITHM (DTQWS)")
print(f"s={s} | threshold={threshold} | max_iter={max_iter}")
print("=" * 70)
print()

for idx, (name, image) in enumerate(all_images.items()):

    print(f"[{idx+1}/{len(all_images)}] {name}.jpg")

    start   = time.time()
    result  = run_dtqws(image, s=s, threshold=threshold, max_iter=max_iter)
    elapsed = time.time() - start

    paper_results[name] = result

    print(f"  M        = {result['M']} ({100*result['M']/result['N']:.2f}% marked)")
    print(f"  peak p_s = {result['best_ps']:.6f}")
    print(f"  peak t   = {result['best_t']}")
    print(f"  runtime  = {elapsed:.1f} sec")
    print()

    # ----------------------------------------
    # Visualization
    # ----------------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"{name}.jpg  |  peak p_s={result['best_ps']:.4f}  |  peak t={result['best_t']}"
    )

    axes[0].imshow(image, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(result['grad'], cmap='hot')
    axes[1].set_title("Gradient Map")
    axes[1].axis('off')

    axes[2].imshow(result['edge_mask'], cmap='gray')
    axes[2].set_title("Marked Set $T_M$ (Eq.3)")
    axes[2].axis('off')

    axes[3].imshow(result['prob_map'], cmap='viridis')
    axes[3].set_title("Probability Map (post-walk)")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

    # convergence plot
    plt.figure(figsize=(8, 4))
    plt.plot(result['ps_history'], linewidth=1.8)
    plt.axvline(
        result['best_t'],
        linestyle='--',
        color='red',
        label=f"peak t={result['best_t']}"
    )
    plt.xlabel("Iteration t")
    plt.ylabel("$p_s$")
    plt.title(f"{name}.jpg — $p_s$ convergence")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

print("=" * 70)
print("PART 1 SUMMARY — peak p_s")
print("=" * 70)
for name, r in paper_results.items():
    print(f"  {name:<15}  p_s={r['best_ps']:.6f}  t={r['best_t']}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 2 — RAW ORACLE EVALUATION
# Tests Eq.(3) edge_mask directly vs BSDS GT
# No quantum walk involved
# ════════════════════════════════════════════

paper_raw_oracle = {}

raw_p, raw_r, raw_f1 = [], [], []

print("=" * 70)
print("PART 2 — RAW ORACLE EVALUATION (Eq.3 mask, no quantum walk)")
print("=" * 70)
print()

for name in paper_results:

    gt           = all_ground_truths[name]
    oracle_mask  = paper_results[name]["edge_mask"].astype(np.uint8)

    p, r, f1 = compute_metrics(oracle_mask, gt)

    paper_raw_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    raw_p.append(p)
    raw_r.append(r)
    raw_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(f"{name}.jpg — Raw Oracle  F1={f1:.4f}")

    axes[0].imshow(all_images[name], cmap='gray')
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(gt, cmap='gray')
    axes[1].set_title("BSDS Ground Truth")
    axes[1].axis('off')

    axes[2].imshow(oracle_mask, cmap='gray')
    axes[2].set_title(f"Oracle Mask  F1={f1:.3f}")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

print()
print("=" * 70)
print("PART 2 SUMMARY — RAW ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(raw_p):.4f}")
print(f"  Recall    = {np.mean(raw_r):.4f}")
print(f"  F1        = {np.mean(raw_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 3 — OTSU ORACLE F1
# Otsu threshold applied to the raw gradient map
# (pre-quantum-walk signal)
# ════════════════════════════════════════════

paper_otsu_oracle = {}

otsu_ora_p, otsu_ora_r, otsu_ora_f1 = [], [], []

print("=" * 70)
print("PART 3 — OTSU ORACLE F1 (Otsu on gradient, pre-walk)")
print("=" * 70)
print()

for name in paper_results:

    gt          = all_ground_truths[name]
    grad        = paper_results[name]["grad"]
    oracle_pred = otsu_edge_map(grad)

    p, r, f1 = compute_metrics(oracle_pred, gt)

    paper_otsu_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_ora_p.append(p)
    otsu_ora_r.append(r)
    otsu_ora_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

print()
print("=" * 70)
print("PART 3 SUMMARY — OTSU ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_ora_p):.4f}")
print(f"  Recall    = {np.mean(otsu_ora_r):.4f}")
print(f"  F1        = {np.mean(otsu_ora_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 4 — OTSU QUANTUM F1
# Otsu threshold applied to the probability map
# (post-quantum-walk signal)
# ════════════════════════════════════════════

paper_otsu_quantum = {}

otsu_qnt_p, otsu_qnt_r, otsu_qnt_f1 = [], [], []

print("=" * 70)
print("PART 4 — OTSU QUANTUM F1 (Otsu on prob_map, post-walk)")
print("=" * 70)
print()

for name in paper_results:

    gt           = all_ground_truths[name]
    prob_map     = paper_results[name]["prob_map"]
    quantum_pred = otsu_edge_map(prob_map)

    p, r, f1 = compute_metrics(quantum_pred, gt)

    paper_otsu_quantum[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_qnt_p.append(p)
    otsu_qnt_r.append(r)
    otsu_qnt_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

    # visualization
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"{name}.jpg  |  Otsu Oracle F1={paper_otsu_oracle[name]['f1']:.4f}"
        f"  |  Otsu Quantum F1={f1:.4f}"
    )

    axes[0].imshow(all_images[name], cmap='gray')
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(all_ground_truths[name], cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')

    axes[2].imshow(
        otsu_edge_map(paper_results[name]["grad"]),
        cmap='gray'
    )
    axes[2].set_title(
        f"Otsu Oracle\nF1={paper_otsu_oracle[name]['f1']:.4f}"
    )
    axes[2].axis('off')

    axes[3].imshow(quantum_pred, cmap='gray')
    axes[3].set_title(f"Otsu Quantum\nF1={f1:.4f}")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

print()
print("=" * 70)
print("PART 4 SUMMARY — OTSU QUANTUM AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_qnt_p):.4f}")
print(f"  Recall    = {np.mean(otsu_qnt_r):.4f}")
print(f"  F1        = {np.mean(otsu_qnt_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# FINAL COMPARISON TABLE
# ════════════════════════════════════════════

print("=" * 70)
print("CELL 2 — FULL EVALUATION SUMMARY (PAPER'S ALGORITHM)")
print("=" * 70)

print(
    f"{'Image':<15}"
    f"{'Raw Oracle F1':<16}"
    f"{'Otsu Oracle F1':<16}"
    f"{'Otsu Quantum F1':<16}"
)
print("-" * 70)

for name in paper_results:
    print(
        f"{name:<15}"
        f"{paper_raw_oracle[name]['f1']:<16.4f}"
        f"{paper_otsu_oracle[name]['f1']:<16.4f}"
        f"{paper_otsu_quantum[name]['f1']:<16.4f}"
    )

print("-" * 70)
print(
    f"{'Average':<15}"
    f"{np.mean(raw_f1):<16.4f}"
    f"{np.mean(otsu_ora_f1):<16.4f}"
    f"{np.mean(otsu_qnt_f1):<16.4f}"
)
print("=" * 70)
print()
print("Stored in:")
print("  paper_results       — DTQWS output (grad, edge_mask, prob_map, p_s)")
print("  paper_raw_oracle    — Raw oracle eval (fixed threshold vs GT)")
print("  paper_otsu_oracle   — Otsu Oracle F1 (Otsu on gradient)")
print("  paper_otsu_quantum  — Otsu Quantum F1 (Otsu on prob_map)")
print()
print("Ready for Cell 3 (diagonal contribution).")

In [ ]:
# ============================================
# CELL 3 — MY CONTRIBUTION + FULL EVALUATION
#
# Contribution: extended gradient computation
# from 4-direction (paper's Eq.3) to 6-direction
# by adding two diagonal neighbours (↘ and ↙).
# Everything else (coin, shift, walk, evaluation)
# is identical to Cell 2.
#
# Contains:
#   PART 1 — DTQWS with diagonal gradient
#   PART 2 — Raw oracle evaluation
#   PART 3 — Otsu Oracle F1 (pre-walk)
#   PART 4 — Otsu Quantum F1 (post-walk)
#   PART 5 — Head-to-head comparison vs paper
#
# Reads  : all_images, all_ground_truths (Cell 1)
#           paper_raw_oracle, paper_otsu_oracle,
#           paper_otsu_quantum (Cell 2)
# Outputs: diagonal_results, diagonal_raw_oracle,
#          diagonal_otsu_oracle, diagonal_otsu_quantum
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import time
from skimage.filters import threshold_otsu

# --------------------------------------------
# PARAMETERS (identical to Cell 2)
# --------------------------------------------
s         = 0.0001
threshold = 30
max_iter  = 1000

# --------------------------------------------
# COIN BASIS
# --------------------------------------------
R, L, U, D, SLF = 0, 1, 2, 3, 4


# ════════════════════════════════════════════
# SHARED HELPERS (identical to Cell 2)
# ════════════════════════════════════════════

def make_coin_vector(s):
    cv = np.array(
        [1, 1, 1, 1, np.sqrt(s)],
        dtype=np.complex128
    )
    cv /= np.linalg.norm(cv)
    return cv

def apply_CG(state, edge_mask, cv):
    out  = state.copy()
    out[edge_mask, SLF] *= -1
    proj = np.einsum('xyc,c->xy', out, cv.conj())
    out  = (
        2.0 * proj[:, :, None] * cv[None, None, :]
        - out
    )
    return out

def apply_shift(state):
    new = np.zeros_like(state)
    new[:, :, L]   += np.roll(state[:, :, R],  1, axis=1)
    new[:, :, R]   += np.roll(state[:, :, L], -1, axis=1)
    new[:, :, D]   += np.roll(state[:, :, U],  1, axis=0)
    new[:, :, U]   += np.roll(state[:, :, D], -1, axis=0)
    new[:, :, SLF] += state[:, :, SLF]
    return new

def compute_metrics(pred, gt):
    TP = np.sum((pred == 1) & (gt == 1))
    FP = np.sum((pred == 1) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )
    return precision, recall, f1

def otsu_edge_map(signal):
    try:
        t = threshold_otsu(signal)
    except ValueError:
        t = signal.max()
    return (signal >= t).astype(np.uint8)


# ════════════════════════════════════════════
# MY CONTRIBUTION — 6-DIRECTION GRADIENT
# Extends Eq.(3) by adding two diagonal
# directions (↘ and ↙) to the max-gradient
# computation. This means a pixel can be marked
# as an edge if it has a strong diagonal
# intensity change, not just axis-aligned.
# ════════════════════════════════════════════

def compute_gradient_diagonal(image):
    """
    Extended gradient: max over 6 directions
    (right, left, down, up, ↘, ↙)
    vs paper's 4 directions (right, left, down, up)
    """

    rows, cols = image.shape
    grad = np.zeros((rows, cols))

    for r in range(rows):
        for c in range(cols):

            center = image[r, c]
            vals   = []

            # 4 axis-aligned directions (paper's Eq.3)
            if c < cols - 1:
                vals.append(abs(center - image[r, c+1]))
            if c > 0:
                vals.append(abs(center - image[r, c-1]))
            if r < rows - 1:
                vals.append(abs(center - image[r+1, c]))
            if r > 0:
                vals.append(abs(center - image[r-1, c]))

            # MY CONTRIBUTION: 2 diagonal directions
            if r < rows - 1 and c < cols - 1:
                vals.append(abs(center - image[r+1, c+1]))  # ↘
            if r < rows - 1 and c > 0:
                vals.append(abs(center - image[r+1, c-1]))  # ↙

            grad[r, c] = max(vals) if vals else 0

    return grad

# --------------------------------------------
# MAIN — run DTQWS with diagonal gradient
# --------------------------------------------
def run_dtqws_diagonal(image, s=0.0001, threshold=30, max_iter=1000):

    rows, cols = image.shape
    N          = rows * cols

    # MY CONTRIBUTION: use diagonal gradient
    grad      = compute_gradient_diagonal(image)
    edge_mask = grad >= threshold
    M         = int(edge_mask.sum())

    cv = make_coin_vector(s)

    coin_amps = np.array(
        [1, 1, 1, 1, np.sqrt(s)],
        dtype=np.complex128
    ) / np.sqrt(4 + s)

    state = np.zeros((rows, cols, 5), dtype=np.complex128)
    for d in range(5):
        state[:, :, d] = coin_amps[d] / np.sqrt(N)

    ps_history = []
    best_ps    = -1
    best_t     = 0
    best_state = state.copy()

    for t in range(1, max_iter + 1):

        state = apply_CG(state, edge_mask, cv)
        state = apply_shift(state)

        ps = float(np.sum(np.abs(state[edge_mask])**2))
        ps_history.append(ps)

        if ps > best_ps:
            best_ps    = ps
            best_t     = t
            best_state = state.copy()

    prob_map = np.sum(np.abs(best_state)**2, axis=2)

    return {
        "grad":       grad,
        "edge_mask":  edge_mask,
        "prob_map":   prob_map,
        "ps_history": ps_history,
        "best_ps":    best_ps,
        "best_t":     best_t,
        "M":          M,
        "N":          N
    }


# ════════════════════════════════════════════
# PART 1 — DTQWS BATCH RUN (DIAGONAL)
# ════════════════════════════════════════════

diagonal_results = {}

print("=" * 70)
print("CELL 3 — MY CONTRIBUTION (DIAGONAL GRADIENT + DTQWS)")
print(f"s={s} | threshold={threshold} | max_iter={max_iter}")
print("Gradient: 6-direction (4 axis-aligned + ↘ + ↙)")
print("=" * 70)
print()

for idx, (name, image) in enumerate(all_images.items()):

    print(f"[{idx+1}/{len(all_images)}] {name}.jpg")

    start   = time.time()
    result  = run_dtqws_diagonal(image, s=s, threshold=threshold, max_iter=max_iter)
    elapsed = time.time() - start

    diagonal_results[name] = result

    print(f"  M        = {result['M']} ({100*result['M']/result['N']:.2f}% marked)")
    print(f"  peak p_s = {result['best_ps']:.6f}")
    print(f"  peak t   = {result['best_t']}")
    print(f"  runtime  = {elapsed:.1f} sec")
    print()

    # ----------------------------------------
    # Visualization
    # ----------------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"{name}.jpg  |  peak p_s={result['best_ps']:.4f}  |  peak t={result['best_t']}"
    )

    axes[0].imshow(image, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(result['grad'], cmap='hot')
    axes[1].set_title("Gradient Map (6-dir)")
    axes[1].axis('off')

    axes[2].imshow(result['edge_mask'], cmap='gray')
    axes[2].set_title("Marked Set $T_M$")
    axes[2].axis('off')

    axes[3].imshow(result['prob_map'], cmap='viridis')
    axes[3].set_title("Probability Map (post-walk)")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

    # convergence plot
    plt.figure(figsize=(8, 4))
    plt.plot(result['ps_history'], linewidth=1.8)
    plt.axvline(
        result['best_t'],
        linestyle='--',
        color='red',
        label=f"peak t={result['best_t']}"
    )
    plt.xlabel("Iteration t")
    plt.ylabel("$p_s$")
    plt.title(f"{name}.jpg — $p_s$ convergence (diagonal)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

print("=" * 70)
print("PART 1 SUMMARY — peak p_s")
print("=" * 70)
for name, r in diagonal_results.items():
    print(f"  {name:<15}  p_s={r['best_ps']:.6f}  t={r['best_t']}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 2 — RAW ORACLE EVALUATION
# ════════════════════════════════════════════

diagonal_raw_oracle = {}

raw_p, raw_r, raw_f1 = [], [], []

print("=" * 70)
print("PART 2 — RAW ORACLE EVALUATION (diagonal mask, no quantum walk)")
print("=" * 70)
print()

for name in diagonal_results:

    gt          = all_ground_truths[name]
    oracle_mask = diagonal_results[name]["edge_mask"].astype(np.uint8)

    p, r, f1 = compute_metrics(oracle_mask, gt)

    diagonal_raw_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    raw_p.append(p)
    raw_r.append(r)
    raw_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(f"{name}.jpg — Diagonal Oracle  F1={f1:.4f}")

    axes[0].imshow(all_images[name], cmap='gray')
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(gt, cmap='gray')
    axes[1].set_title("BSDS Ground Truth")
    axes[1].axis('off')

    axes[2].imshow(oracle_mask, cmap='gray')
    axes[2].set_title(f"Diagonal Oracle Mask  F1={f1:.3f}")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

print()
print("=" * 70)
print("PART 2 SUMMARY — RAW ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(raw_p):.4f}")
print(f"  Recall    = {np.mean(raw_r):.4f}")
print(f"  F1        = {np.mean(raw_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 3 — OTSU ORACLE F1
# ════════════════════════════════════════════

diagonal_otsu_oracle = {}

otsu_ora_p, otsu_ora_r, otsu_ora_f1 = [], [], []

print("=" * 70)
print("PART 3 — OTSU ORACLE F1 (Otsu on diagonal gradient, pre-walk)")
print("=" * 70)
print()

for name in diagonal_results:

    gt          = all_ground_truths[name]
    grad        = diagonal_results[name]["grad"]
    oracle_pred = otsu_edge_map(grad)

    p, r, f1 = compute_metrics(oracle_pred, gt)

    diagonal_otsu_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_ora_p.append(p)
    otsu_ora_r.append(r)
    otsu_ora_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

print()
print("=" * 70)
print("PART 3 SUMMARY — OTSU ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_ora_p):.4f}")
print(f"  Recall    = {np.mean(otsu_ora_r):.4f}")
print(f"  F1        = {np.mean(otsu_ora_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 4 — OTSU QUANTUM F1
# ════════════════════════════════════════════

diagonal_otsu_quantum = {}

otsu_qnt_p, otsu_qnt_r, otsu_qnt_f1 = [], [], []

print("=" * 70)
print("PART 4 — OTSU QUANTUM F1 (Otsu on prob_map, post-walk)")
print("=" * 70)
print()

for name in diagonal_results:

    gt           = all_ground_truths[name]
    prob_map     = diagonal_results[name]["prob_map"]
    quantum_pred = otsu_edge_map(prob_map)

    p, r, f1 = compute_metrics(quantum_pred, gt)

    diagonal_otsu_quantum[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_qnt_p.append(p)
    otsu_qnt_r.append(r)
    otsu_qnt_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"{name}.jpg  |  Otsu Oracle F1={diagonal_otsu_oracle[name]['f1']:.4f}"
        f"  |  Otsu Quantum F1={f1:.4f}"
    )

    axes[0].imshow(all_images[name], cmap='gray')
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(all_ground_truths[name], cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')

    axes[2].imshow(
        otsu_edge_map(diagonal_results[name]["grad"]),
        cmap='gray'
    )
    axes[2].set_title(
        f"Otsu Oracle\nF1={diagonal_otsu_oracle[name]['f1']:.4f}"
    )
    axes[2].axis('off')

    axes[3].imshow(quantum_pred, cmap='gray')
    axes[3].set_title(f"Otsu Quantum\nF1={f1:.4f}")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

print()
print("=" * 70)
print("PART 4 SUMMARY — OTSU QUANTUM AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_qnt_p):.4f}")
print(f"  Recall    = {np.mean(otsu_qnt_r):.4f}")
print(f"  F1        = {np.mean(otsu_qnt_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 5 — HEAD-TO-HEAD vs PAPER
# ════════════════════════════════════════════

print("=" * 70)
print("PART 5 — HEAD-TO-HEAD: PAPER vs MY CONTRIBUTION (DIAGONAL)")
print("=" * 70)

print(
    f"\n{'Metric':<25}"
    f"{'Paper':<12}"
    f"{'Diagonal':<12}"
    f"{'Δ':<10}"
)
print("-" * 60)

comparisons = [
    (
        "Raw Oracle F1",
        np.mean([paper_raw_oracle[n]['f1'] for n in paper_raw_oracle]),
        np.mean(raw_f1)
    ),
    (
        "Otsu Oracle F1",
        np.mean([paper_otsu_oracle[n]['f1'] for n in paper_otsu_oracle]),
        np.mean(otsu_ora_f1)
    ),
    (
        "Otsu Quantum F1",
        np.mean([paper_otsu_quantum[n]['f1'] for n in paper_otsu_quantum]),
        np.mean(otsu_qnt_f1)
    ),
]

for label, paper_val, diag_val in comparisons:
    delta  = diag_val - paper_val
    symbol = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(
        f"  {label:<23}"
        f"{paper_val:<12.4f}"
        f"{diag_val:<12.4f}"
        f"{symbol}{abs(delta):.4f}"
    )

print("-" * 60)
print()

# per-image table
print(
    f"{'Image':<15}"
    f"{'Raw Oracle F1':<16}"
    f"{'Otsu Oracle F1':<16}"
    f"{'Otsu Quantum F1':<16}"
)
print("-" * 70)

for name in diagonal_results:
    print(
        f"{name:<15}"
        f"{diagonal_raw_oracle[name]['f1']:<16.4f}"
        f"{diagonal_otsu_oracle[name]['f1']:<16.4f}"
        f"{diagonal_otsu_quantum[name]['f1']:<16.4f}"
    )

print("-" * 70)
print(
    f"{'Average':<15}"
    f"{np.mean(raw_f1):<16.4f}"
    f"{np.mean(otsu_ora_f1):<16.4f}"
    f"{np.mean(otsu_qnt_f1):<16.4f}"
)
print("=" * 70)
print()
print("Stored in:")
print("  diagonal_results       — DTQWS output (grad, edge_mask, prob_map, p_s)")
print("  diagonal_raw_oracle    — Raw oracle eval")
print("  diagonal_otsu_oracle   — Otsu Oracle F1")
print("  diagonal_otsu_quantum  — Otsu Quantum F1")

In [ ]:
# ════════════════════════════════════════════
# SAVE THESIS IMAGES FOR 29030 AND 8068
# Run this after PART 4 completes
# ════════════════════════════════════════════

from skimage.filters import threshold_otsu
from google.colab import files
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print("Saving thesis images...")

for name in ['8068', '29030']:

    img    = all_images[name]
    result = diagonal_results[name]
    gt     = all_ground_truths[name]

    # (a) original
    plt.imsave(f'{name}_original.jpg', img, cmap='gray')

    # (b) ground truth
    plt.imsave(f'{name}_gt.png', gt, cmap='gray')

    # (c) gradient map
    plt.imsave(f'{name}_gradient.png', result['grad'], cmap='hot')

    # (d) marked set T_M
    plt.imsave(
        f'{name}_edge_mask.png',
        result['edge_mask'].astype(float),
        cmap='gray'
    )

    # (e) probability map
    plt.imsave(f'{name}_prob_map.png', result['prob_map'], cmap='viridis')

    # (f) otsu quantum edge map
    try:
        t_q = threshold_otsu(result['prob_map'])
    except ValueError:
        t_q = result['prob_map'].max()
    otsu_q = (result['prob_map'] >= t_q).astype(float)
    plt.imsave(f'{name}_otsu_quantum.png', otsu_q, cmap='gray')

    # (g) otsu oracle edge map
    try:
        t_o = threshold_otsu(result['grad'])
    except ValueError:
        t_o = result['grad'].max()
    otsu_o = (result['grad'] >= t_o).astype(float)
    plt.imsave(f'{name}_otsu_oracle.png', otsu_o, cmap='gray')

    # (h) convergence plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(result['ps_history'], linewidth=1.8)
    ax.axvline(
        result['best_t'],
        linestyle='--',
        color='red',
        label=f"peak t={result['best_t']}"
    )
    ax.set_xlabel('Iteration t')
    ax.set_ylabel('$p_s$')
    ax.set_title(f'{name} — $p_s$ convergence (diagonal)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{name}_convergence.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f"  Saved all 8 images for {name}")

# ---- Download all 16 files ----
print()
print("Downloading...")

for name in ['29030', '8068']:
    for suffix in [
        'original.jpg',
        'gt.png',
        'gradient.png',
        'edge_mask.png',
        'prob_map.png',
        'otsu_quantum.png',
        'otsu_oracle.png',
        'convergence.png'
    ]:
        files.download(f'{name}_{suffix}')

print("Done. Upload all 16 files into an images/ folder in Overleaf.")

In [ ]:
from google.colab import files
for suffix in [
    'gradient.png',
    'edge_mask.png',
    'prob_map.png',
    'otsu_quantum.png',
    'otsu_oracle.png',
    'convergence.png'
]:
    files.download(f'8068_{suffix}')

# ***QASM SIMULATOR***

In [ ]:
# ============================================
# INSTALL QISKIT
# ============================================

!pip install qiskit qiskit-aer qiskit-ibm-runtime

# verify installation
import qiskit
import qiskit_aer
print(f"Qiskit version     : {qiskit.__version__}")
print(f"Qiskit Aer version : {qiskit_aer.__version__}")
print("Installation successful. Ready for circuit implementation.")

In [ ]:
# ============================================
# CELL 2 — PAPER'S ALGORITHM ON QASM SIMULATOR
#
# Implements the paper's Section III block-based
# quantum circuit approach using Qiskit's
# qasm_simulator (AerSimulator).
#
# KEY TRICK: a 2x2 block circuit depends only
# on which of its 4 pixels are marked (16
# possible patterns). So we run the qasm
# simulator just 16 times total (one per
# pattern), build a lookup table, then assign
# results to every block by pattern match.
# This turns ~hours into ~30 seconds total.
#
# Contains:
#   PART 1 — Build 16-pattern lookup table
#   PART 2 — Process all images via lookup
#   PART 3 — Raw oracle evaluation
#   PART 4 — Otsu Oracle F1 (pre-walk)
#   PART 5 — Otsu Quantum F1 (post-walk)
#   PART 6 — Full summary table
#
# Reads  : all_images, all_ground_truths (Cell 1)
# Outputs: qasm_paper_results,
#          qasm_paper_raw_oracle,
#          qasm_paper_otsu_oracle,
#          qasm_paper_otsu_quantum
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import time
from skimage.filters import threshold_otsu
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator

# --------------------------------------------
# PARAMETERS
# --------------------------------------------
s_param   = 0.1    # self-loop weight (paper Fig.2)
t_param   = 5      # walk iterations  (paper Fig.2)
threshold = 30     # gradient threshold a_th (Eq.3)
shots     = 20000  # qasm shots per pattern

# --------------------------------------------
# COIN BASIS (4-vertex cycle block)
# --------------------------------------------
BL, BR, BS1, BS2 = 0, 1, 2, 3


# ════════════════════════════════════════════
# CIRCUIT CONSTRUCTION
# ════════════════════════════════════════════

def make_block_coin_vector(s):
    cv = np.array(
        [1, 1, np.sqrt(s), np.sqrt(s)],
        dtype=np.complex128
    )
    cv /= np.linalg.norm(cv)
    return cv

def build_CG_matrix(marked_vertices, s):

    cv = make_block_coin_vector(s)
    C  = (
        2.0 * np.outer(cv, cv.conj())
        - np.eye(4, dtype=np.complex128)
    )

    CG = np.zeros((16, 16), dtype=np.complex128)

    for v in range(4):
        if v in marked_vertices:
            oracle = np.diag(
                [1, 1, -1, -1]
            ).astype(np.complex128)
            block = C @ oracle
        else:
            block = C
        CG[v*4:(v+1)*4, v*4:(v+1)*4] = block

    return CG

def build_S_matrix():

    S = np.zeros((16, 16), dtype=np.complex128)

    for v in range(4):
        # Right -> v+1, becomes Left
        S[((v+1) % 4)*4 + BL, v*4 + BR] = 1.0
        # Left -> v-1, becomes Right
        S[((v-1) % 4)*4 + BR, v*4 + BL] = 1.0
        # self-loops stay
        S[v*4 + BS1, v*4 + BS1] = 1.0
        S[v*4 + BS2, v*4 + BS2] = 1.0

    return S

def build_block_circuit(marked_vertices, s, t):

    CG   = build_CG_matrix(marked_vertices, s)
    Smat = build_S_matrix()

    CG_gate = UnitaryGate(CG, label="CG")
    S_gate  = UnitaryGate(Smat, label="S")

    qc = QuantumCircuit(4, 4)

    # vertex register: uniform superposition
    qc.h(0)
    qc.h(1)

    # coin register: paper's coin state
    cv = make_block_coin_vector(s)
    qc.initialize(cv, [2, 3])

    for _ in range(t):
        qc.append(CG_gate, [2, 3, 0, 1])
        qc.append(S_gate,  [2, 3, 0, 1])

    qc.measure([0, 1, 2, 3], [0, 1, 2, 3])

    return qc


# ════════════════════════════════════════════
# PART 1 — BUILD 16-PATTERN LOOKUP TABLE
# ════════════════════════════════════════════

print("=" * 70)
print("CELL 2 — PAPER'S ALGORITHM ON QASM SIMULATOR")
print(f"s={s_param} | t={t_param} | shots={shots} | threshold={threshold}")
print("=" * 70)
print()
print("PART 1 — Building 16-pattern lookup table...")

t0  = time.time()
sim = AerSimulator()

circuits = []
for bitmask in range(16):
    marked = [v for v in range(4) if (bitmask >> v) & 1]
    circuits.append(build_block_circuit(marked, s_param, t_param))

compiled = transpile(circuits, sim)
job      = sim.run(compiled, shots=shots)
result   = job.result()

qasm_paper_lookup = {}
for bitmask in range(16):
    counts = result.get_counts(bitmask)
    vc = {0: 0, 1: 0, 2: 0, 3: 0}
    for bitstring, n in counts.items():
        v = int(bitstring[-2:], 2)
        vc[v] += n
    total = sum(vc.values())
    qasm_paper_lookup[bitmask] = np.array(
        [vc[v] / total for v in range(4)]
    )

print(f"Lookup table built in {time.time()-t0:.2f} sec")
print()


# ════════════════════════════════════════════
# SHARED HELPERS
# ════════════════════════════════════════════

def compute_gradient(image):
    """4-direction gradient (paper Eq.3)"""
    rows, cols = image.shape
    grad = np.zeros((rows, cols))
    for r in range(rows):
        for c in range(cols):
            center = image[r, c]
            vals   = []
            if c < cols - 1: vals.append(abs(center - image[r, c+1]))
            if c > 0:         vals.append(abs(center - image[r, c-1]))
            if r < rows - 1: vals.append(abs(center - image[r+1, c]))
            if r > 0:         vals.append(abs(center - image[r-1, c]))
            grad[r, c] = max(vals) if vals else 0
    return grad

def compute_metrics(pred, gt):
    TP = np.sum((pred == 1) & (gt == 1))
    FP = np.sum((pred == 1) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )
    return precision, recall, f1

def otsu_edge_map(signal):
    try:
        t = threshold_otsu(signal)
    except ValueError:
        t = signal.max()
    return (signal >= t).astype(np.uint8)

def run_qasm_image(image, lookup, threshold=30):
    """Process one image using the lookup table."""

    orig_rows, orig_cols = image.shape

    # pad to even dimensions if needed
    pad_r = orig_rows % 2
    pad_c = orig_cols % 2
    if pad_r or pad_c:
        image = np.pad(
            image,
            ((0, pad_r), (0, pad_c)),
            mode='edge'
        )

    rows, cols = image.shape

    grad      = compute_gradient(image)
    edge_mask = grad >= threshold
    prob_map  = np.zeros((rows, cols))

    for br in range(0, rows, 2):
        for bc in range(0, cols, 2):

            m = [
                edge_mask[br,   bc],
                edge_mask[br,   bc+1],
                edge_mask[br+1, bc+1],
                edge_mask[br+1, bc],
            ]

            bitmask = sum(
                (1 << v) for v in range(4) if m[v]
            )

            probs = lookup[bitmask]

            prob_map[br,   bc]   = probs[0]
            prob_map[br,   bc+1] = probs[1]
            prob_map[br+1, bc+1] = probs[2]
            prob_map[br+1, bc]   = probs[3]

    # crop back to original size
    grad      = grad[:orig_rows, :orig_cols]
    edge_mask = edge_mask[:orig_rows, :orig_cols]
    prob_map  = prob_map[:orig_rows, :orig_cols]

    return {
        "grad":      grad,
        "edge_mask": edge_mask,
        "prob_map":  prob_map,
        "M":         int(edge_mask.sum()),
        "N":         orig_rows * orig_cols
    }


# ════════════════════════════════════════════
# PART 2 — PROCESS ALL IMAGES
# ════════════════════════════════════════════

qasm_paper_results = {}

print("=" * 70)
print("PART 2 — PROCESSING ALL IMAGES")
print("=" * 70)
print()

for idx, (name, image) in enumerate(all_images.items()):

    print(f"[{idx+1}/{len(all_images)}] {name}.jpg")

    start  = time.time()
    result = run_qasm_image(image, qasm_paper_lookup, threshold=threshold)
    elapsed = time.time() - start

    qasm_paper_results[name] = result

    print(f"  M={result['M']} ({100*result['M']/result['N']:.2f}% marked)")
    print(f"  runtime={elapsed:.3f} sec")
    print()

    # visualization
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f"{name}.jpg | QASM Paper (s={s_param}, t={t_param})")

    axes[0].imshow(image, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(result['grad'], cmap='hot')
    axes[1].set_title("Gradient Map")
    axes[1].axis('off')

    axes[2].imshow(result['edge_mask'], cmap='gray')
    axes[2].set_title("Marked Set $T_M$")
    axes[2].axis('off')

    axes[3].imshow(result['prob_map'], cmap='viridis')
    axes[3].set_title("Probability Map")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

print("=" * 70)
print("PART 2 SUMMARY — M (marked pixels)")
print("=" * 70)
for name, r in qasm_paper_results.items():
    print(f"  {name:<15}  M={r['M']} ({100*r['M']/r['N']:.2f}%)")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 3 — RAW ORACLE EVALUATION
# ════════════════════════════════════════════

qasm_paper_raw_oracle = {}

raw_p, raw_r, raw_f1 = [], [], []

print("=" * 70)
print("PART 3 — RAW ORACLE EVALUATION (Eq.3 mask, no quantum circuit)")
print("=" * 70)
print()

for name in qasm_paper_results:

    gt          = all_ground_truths[name]
    oracle_mask = qasm_paper_results[name]["edge_mask"].astype(np.uint8)

    p, r, f1 = compute_metrics(oracle_mask, gt)

    qasm_paper_raw_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    raw_p.append(p)
    raw_r.append(r)
    raw_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

print()
print("=" * 70)
print("PART 3 SUMMARY — RAW ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(raw_p):.4f}")
print(f"  Recall    = {np.mean(raw_r):.4f}")
print(f"  F1        = {np.mean(raw_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 4 — OTSU ORACLE F1
# Otsu on raw gradient (pre-circuit signal)
# ════════════════════════════════════════════

qasm_paper_otsu_oracle = {}

otsu_ora_p, otsu_ora_r, otsu_ora_f1 = [], [], []

print("=" * 70)
print("PART 4 — OTSU ORACLE F1 (Otsu on gradient, pre-circuit)")
print("=" * 70)
print()

for name in qasm_paper_results:

    gt          = all_ground_truths[name]
    grad        = qasm_paper_results[name]["grad"]
    oracle_pred = otsu_edge_map(grad)

    p, r, f1 = compute_metrics(oracle_pred, gt)

    qasm_paper_otsu_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_ora_p.append(p)
    otsu_ora_r.append(r)
    otsu_ora_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

print()
print("=" * 70)
print("PART 4 SUMMARY — OTSU ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_ora_p):.4f}")
print(f"  Recall    = {np.mean(otsu_ora_r):.4f}")
print(f"  F1        = {np.mean(otsu_ora_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 5 — OTSU QUANTUM F1
# Otsu on probability map (post-circuit signal)
# ════════════════════════════════════════════

qasm_paper_otsu_quantum = {}

otsu_qnt_p, otsu_qnt_r, otsu_qnt_f1 = [], [], []

print("=" * 70)
print("PART 5 — OTSU QUANTUM F1 (Otsu on prob_map, post-circuit)")
print("=" * 70)
print()

for name in qasm_paper_results:

    gt           = all_ground_truths[name]
    prob_map     = qasm_paper_results[name]["prob_map"]
    quantum_pred = otsu_edge_map(prob_map)

    p, r, f1 = compute_metrics(quantum_pred, gt)

    qasm_paper_otsu_quantum[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_qnt_p.append(p)
    otsu_qnt_r.append(r)
    otsu_qnt_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

    # visualization
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"{name}.jpg  |  Otsu Oracle F1={qasm_paper_otsu_oracle[name]['f1']:.4f}"
        f"  |  Otsu Quantum F1={f1:.4f}"
    )

    axes[0].imshow(all_images[name], cmap='gray')
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(all_ground_truths[name], cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')

    axes[2].imshow(
        otsu_edge_map(qasm_paper_results[name]["grad"]),
        cmap='gray'
    )
    axes[2].set_title(
        f"Otsu Oracle\nF1={qasm_paper_otsu_oracle[name]['f1']:.4f}"
    )
    axes[2].axis('off')

    axes[3].imshow(quantum_pred, cmap='gray')
    axes[3].set_title(f"Otsu Quantum\nF1={f1:.4f}")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

print()
print("=" * 70)
print("PART 5 SUMMARY — OTSU QUANTUM AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_qnt_p):.4f}")
print(f"  Recall    = {np.mean(otsu_qnt_r):.4f}")
print(f"  F1        = {np.mean(otsu_qnt_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 6 — FULL SUMMARY TABLE
# ════════════════════════════════════════════

print("=" * 70)
print("CELL 2 — FULL EVALUATION SUMMARY (QASM PAPER)")
print("=" * 70)

print(
    f"{'Image':<15}"
    f"{'Raw Oracle F1':<16}"
    f"{'Otsu Oracle F1':<16}"
    f"{'Otsu Quantum F1':<16}"
)
print("-" * 70)

for name in qasm_paper_results:
    print(
        f"{name:<15}"
        f"{qasm_paper_raw_oracle[name]['f1']:<16.4f}"
        f"{qasm_paper_otsu_oracle[name]['f1']:<16.4f}"
        f"{qasm_paper_otsu_quantum[name]['f1']:<16.4f}"
    )

print("-" * 70)
print(
    f"{'Average':<15}"
    f"{np.mean(raw_f1):<16.4f}"
    f"{np.mean(otsu_ora_f1):<16.4f}"
    f"{np.mean(otsu_qnt_f1):<16.4f}"
)
print("=" * 70)
print()
print("Stored in:")
print("  qasm_paper_results       — prob_map, grad, edge_mask per image")
print("  qasm_paper_raw_oracle    — Raw oracle eval")
print("  qasm_paper_otsu_oracle   — Otsu Oracle F1")
print("  qasm_paper_otsu_quantum  — Otsu Quantum F1")
print()
print("Ready for Cell 3 (diagonal contribution on qasm simulator).")

In [ ]:
# ============================================
# CELL 3 — DIAGONAL CONTRIBUTION ON QASM SIMULATOR
#
# My contribution: extended gradient from
# 4-direction (paper Eq.3) to 6-direction
# (adds ↘ and ↙ diagonal neighbours).
# Everything else identical to Cell 2.
#
# Contains:
#   PART 1 — Build 16-pattern lookup table
#             (diagonal gradient for marking)
#   PART 2 — Process all images via lookup
#   PART 3 — Raw oracle evaluation
#   PART 4 — Otsu Oracle F1 (pre-circuit)
#   PART 5 — Otsu Quantum F1 (post-circuit)
#   PART 6 — Full summary table
#   PART 7 — Head-to-head vs QASM paper (Cell 2)
#
# Reads  : all_images, all_ground_truths (Cell 1)
#           qasm_paper_raw_oracle,
#           qasm_paper_otsu_oracle,
#           qasm_paper_otsu_quantum (Cell 2)
# Outputs: qasm_diag_results,
#          qasm_diag_raw_oracle,
#          qasm_diag_otsu_oracle,
#          qasm_diag_otsu_quantum
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import time
from skimage.filters import threshold_otsu
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator

# --------------------------------------------
# PARAMETERS (identical to Cell 2)
# --------------------------------------------
s_param   = 0.1
t_param   = 5
threshold = 30
shots     = 20000

# --------------------------------------------
# COIN BASIS (4-vertex cycle block)
# --------------------------------------------
BL, BR, BS1, BS2 = 0, 1, 2, 3


# ════════════════════════════════════════════
# CIRCUIT CONSTRUCTION (identical to Cell 2)
# ════════════════════════════════════════════

def make_block_coin_vector(s):
    cv = np.array(
        [1, 1, np.sqrt(s), np.sqrt(s)],
        dtype=np.complex128
    )
    cv /= np.linalg.norm(cv)
    return cv

def build_CG_matrix(marked_vertices, s):

    cv = make_block_coin_vector(s)
    C  = (
        2.0 * np.outer(cv, cv.conj())
        - np.eye(4, dtype=np.complex128)
    )

    CG = np.zeros((16, 16), dtype=np.complex128)

    for v in range(4):
        if v in marked_vertices:
            oracle = np.diag(
                [1, 1, -1, -1]
            ).astype(np.complex128)
            block = C @ oracle
        else:
            block = C
        CG[v*4:(v+1)*4, v*4:(v+1)*4] = block

    return CG

def build_S_matrix():

    S = np.zeros((16, 16), dtype=np.complex128)

    for v in range(4):
        S[((v+1) % 4)*4 + BL, v*4 + BR] = 1.0
        S[((v-1) % 4)*4 + BR, v*4 + BL] = 1.0
        S[v*4 + BS1, v*4 + BS1] = 1.0
        S[v*4 + BS2, v*4 + BS2] = 1.0

    return S

def build_block_circuit(marked_vertices, s, t):

    CG   = build_CG_matrix(marked_vertices, s)
    Smat = build_S_matrix()

    CG_gate = UnitaryGate(CG, label="CG")
    S_gate  = UnitaryGate(Smat, label="S")

    qc = QuantumCircuit(4, 4)
    qc.h(0)
    qc.h(1)

    cv = make_block_coin_vector(s)
    qc.initialize(cv, [2, 3])

    for _ in range(t):
        qc.append(CG_gate, [2, 3, 0, 1])
        qc.append(S_gate,  [2, 3, 0, 1])

    qc.measure([0, 1, 2, 3], [0, 1, 2, 3])

    return qc


# ════════════════════════════════════════════
# PART 1 — BUILD 16-PATTERN LOOKUP TABLE
# (uses diagonal gradient to decide marking,
# but the quantum circuit itself is identical
# to Cell 2 — only the marking changes)
# ════════════════════════════════════════════

print("=" * 70)
print("CELL 3 — DIAGONAL CONTRIBUTION ON QASM SIMULATOR")
print(f"s={s_param} | t={t_param} | shots={shots} | threshold={threshold}")
print("Gradient: 6-direction (4 axis-aligned + ↘ + ↙)")
print("=" * 70)
print()
print("PART 1 — Building 16-pattern lookup table...")

t0  = time.time()
sim = AerSimulator()

circuits = []
for bitmask in range(16):
    marked = [v for v in range(4) if (bitmask >> v) & 1]
    circuits.append(build_block_circuit(marked, s_param, t_param))

compiled = transpile(circuits, sim)
job      = sim.run(compiled, shots=shots)
result   = job.result()

qasm_diag_lookup = {}
for bitmask in range(16):
    counts = result.get_counts(bitmask)
    vc = {0: 0, 1: 0, 2: 0, 3: 0}
    for bitstring, n in counts.items():
        v = int(bitstring[-2:], 2)
        vc[v] += n
    total = sum(vc.values())
    qasm_diag_lookup[bitmask] = np.array(
        [vc[v] / total for v in range(4)]
    )

print(f"Lookup table built in {time.time()-t0:.2f} sec")
print()


# ════════════════════════════════════════════
# SHARED HELPERS
# ════════════════════════════════════════════

def compute_gradient_diagonal(image):
    """
    6-direction gradient (MY CONTRIBUTION):
    extends paper's Eq.3 by adding ↘ and ↙
    """
    rows, cols = image.shape
    grad = np.zeros((rows, cols))

    for r in range(rows):
        for c in range(cols):

            center = image[r, c]
            vals   = []

            # 4 axis-aligned (paper's Eq.3)
            if c < cols - 1: vals.append(abs(center - image[r, c+1]))
            if c > 0:         vals.append(abs(center - image[r, c-1]))
            if r < rows - 1: vals.append(abs(center - image[r+1, c]))
            if r > 0:         vals.append(abs(center - image[r-1, c]))

            # MY CONTRIBUTION: 2 diagonal directions
            if r < rows - 1 and c < cols - 1:
                vals.append(abs(center - image[r+1, c+1]))  # ↘
            if r < rows - 1 and c > 0:
                vals.append(abs(center - image[r+1, c-1]))  # ↙

            grad[r, c] = max(vals) if vals else 0

    return grad

def compute_metrics(pred, gt):
    TP = np.sum((pred == 1) & (gt == 1))
    FP = np.sum((pred == 1) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )
    return precision, recall, f1

def otsu_edge_map(signal):
    try:
        t = threshold_otsu(signal)
    except ValueError:
        t = signal.max()
    return (signal >= t).astype(np.uint8)

def run_qasm_image_diagonal(image, lookup, threshold=30):
    """Process one image using diagonal gradient + lookup table."""

    orig_rows, orig_cols = image.shape

    pad_r = orig_rows % 2
    pad_c = orig_cols % 2
    if pad_r or pad_c:
        image = np.pad(
            image,
            ((0, pad_r), (0, pad_c)),
            mode='edge'
        )

    rows, cols = image.shape

    # MY CONTRIBUTION: diagonal gradient for marking
    grad      = compute_gradient_diagonal(image)
    edge_mask = grad >= threshold
    prob_map  = np.zeros((rows, cols))

    for br in range(0, rows, 2):
        for bc in range(0, cols, 2):

            m = [
                edge_mask[br,   bc],
                edge_mask[br,   bc+1],
                edge_mask[br+1, bc+1],
                edge_mask[br+1, bc],
            ]

            bitmask = sum(
                (1 << v) for v in range(4) if m[v]
            )

            probs = lookup[bitmask]

            prob_map[br,   bc]   = probs[0]
            prob_map[br,   bc+1] = probs[1]
            prob_map[br+1, bc+1] = probs[2]
            prob_map[br+1, bc]   = probs[3]

    # crop back to original size
    grad      = grad[:orig_rows, :orig_cols]
    edge_mask = edge_mask[:orig_rows, :orig_cols]
    prob_map  = prob_map[:orig_rows, :orig_cols]

    return {
        "grad":      grad,
        "edge_mask": edge_mask,
        "prob_map":  prob_map,
        "M":         int(edge_mask.sum()),
        "N":         orig_rows * orig_cols
    }


# ════════════════════════════════════════════
# PART 2 — PROCESS ALL IMAGES
# ════════════════════════════════════════════

qasm_diag_results = {}

print("=" * 70)
print("PART 2 — PROCESSING ALL IMAGES (diagonal gradient)")
print("=" * 70)
print()

for idx, (name, image) in enumerate(all_images.items()):

    print(f"[{idx+1}/{len(all_images)}] {name}.jpg")

    start   = time.time()
    result  = run_qasm_image_diagonal(
        image, qasm_diag_lookup, threshold=threshold
    )
    elapsed = time.time() - start

    qasm_diag_results[name] = result

    print(f"  M={result['M']} ({100*result['M']/result['N']:.2f}% marked)")
    print(f"  runtime={elapsed:.3f} sec")
    print()

    # visualization
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"{name}.jpg | QASM Diagonal (s={s_param}, t={t_param})"
    )

    axes[0].imshow(image, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(result['grad'], cmap='hot')
    axes[1].set_title("Gradient Map (6-dir)")
    axes[1].axis('off')

    axes[2].imshow(result['edge_mask'], cmap='gray')
    axes[2].set_title("Marked Set $T_M$")
    axes[2].axis('off')

    axes[3].imshow(result['prob_map'], cmap='viridis')
    axes[3].set_title("Probability Map")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

print("=" * 70)
print("PART 2 SUMMARY — M (marked pixels)")
print("=" * 70)
for name, r in qasm_diag_results.items():
    print(f"  {name:<15}  M={r['M']} ({100*r['M']/r['N']:.2f}%)")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 3 — RAW ORACLE EVALUATION
# ════════════════════════════════════════════

qasm_diag_raw_oracle = {}

raw_p, raw_r, raw_f1 = [], [], []

print("=" * 70)
print("PART 3 — RAW ORACLE EVALUATION (diagonal mask, no circuit)")
print("=" * 70)
print()

for name in qasm_diag_results:

    gt          = all_ground_truths[name]
    oracle_mask = qasm_diag_results[name]["edge_mask"].astype(np.uint8)

    p, r, f1 = compute_metrics(oracle_mask, gt)

    qasm_diag_raw_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    raw_p.append(p)
    raw_r.append(r)
    raw_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

print()
print("=" * 70)
print("PART 3 SUMMARY — RAW ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(raw_p):.4f}")
print(f"  Recall    = {np.mean(raw_r):.4f}")
print(f"  F1        = {np.mean(raw_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 4 — OTSU ORACLE F1
# ════════════════════════════════════════════

qasm_diag_otsu_oracle = {}

otsu_ora_p, otsu_ora_r, otsu_ora_f1 = [], [], []

print("=" * 70)
print("PART 4 — OTSU ORACLE F1 (Otsu on diagonal gradient, pre-circuit)")
print("=" * 70)
print()

for name in qasm_diag_results:

    gt          = all_ground_truths[name]
    grad        = qasm_diag_results[name]["grad"]
    oracle_pred = otsu_edge_map(grad)

    p, r, f1 = compute_metrics(oracle_pred, gt)

    qasm_diag_otsu_oracle[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_ora_p.append(p)
    otsu_ora_r.append(r)
    otsu_ora_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

print()
print("=" * 70)
print("PART 4 SUMMARY — OTSU ORACLE AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_ora_p):.4f}")
print(f"  Recall    = {np.mean(otsu_ora_r):.4f}")
print(f"  F1        = {np.mean(otsu_ora_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 5 — OTSU QUANTUM F1
# ════════════════════════════════════════════

qasm_diag_otsu_quantum = {}

otsu_qnt_p, otsu_qnt_r, otsu_qnt_f1 = [], [], []

print("=" * 70)
print("PART 5 — OTSU QUANTUM F1 (Otsu on prob_map, post-circuit)")
print("=" * 70)
print()

for name in qasm_diag_results:

    gt           = all_ground_truths[name]
    prob_map     = qasm_diag_results[name]["prob_map"]
    quantum_pred = otsu_edge_map(prob_map)

    p, r, f1 = compute_metrics(quantum_pred, gt)

    qasm_diag_otsu_quantum[name] = {"precision": p, "recall": r, "f1": f1}

    otsu_qnt_p.append(p)
    otsu_qnt_r.append(r)
    otsu_qnt_f1.append(f1)

    print(f"  {name}: P={p:.4f}  R={r:.4f}  F1={f1:.4f}")

    # visualization
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"{name}.jpg  |  Otsu Oracle F1={qasm_diag_otsu_oracle[name]['f1']:.4f}"
        f"  |  Otsu Quantum F1={f1:.4f}"
    )

    axes[0].imshow(all_images[name], cmap='gray')
    axes[0].set_title("Original")
    axes[0].axis('off')

    axes[1].imshow(all_ground_truths[name], cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')

    axes[2].imshow(
        otsu_edge_map(qasm_diag_results[name]["grad"]),
        cmap='gray'
    )
    axes[2].set_title(
        f"Otsu Oracle\nF1={qasm_diag_otsu_oracle[name]['f1']:.4f}"
    )
    axes[2].axis('off')

    axes[3].imshow(quantum_pred, cmap='gray')
    axes[3].set_title(f"Otsu Quantum\nF1={f1:.4f}")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

print()
print("=" * 70)
print("PART 5 SUMMARY — OTSU QUANTUM AVERAGE")
print("=" * 70)
print(f"  Precision = {np.mean(otsu_qnt_p):.4f}")
print(f"  Recall    = {np.mean(otsu_qnt_r):.4f}")
print(f"  F1        = {np.mean(otsu_qnt_f1):.4f}")
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 6 — FULL SUMMARY TABLE
# ════════════════════════════════════════════

print("=" * 70)
print("CELL 3 — FULL EVALUATION SUMMARY (QASM DIAGONAL)")
print("=" * 70)

print(
    f"{'Image':<15}"
    f"{'Raw Oracle F1':<16}"
    f"{'Otsu Oracle F1':<16}"
    f"{'Otsu Quantum F1':<16}"
)
print("-" * 70)

for name in qasm_diag_results:
    print(
        f"{name:<15}"
        f"{qasm_diag_raw_oracle[name]['f1']:<16.4f}"
        f"{qasm_diag_otsu_oracle[name]['f1']:<16.4f}"
        f"{qasm_diag_otsu_quantum[name]['f1']:<16.4f}"
    )

print("-" * 70)
print(
    f"{'Average':<15}"
    f"{np.mean(raw_f1):<16.4f}"
    f"{np.mean(otsu_ora_f1):<16.4f}"
    f"{np.mean(otsu_qnt_f1):<16.4f}"
)
print("=" * 70)
print()


# ════════════════════════════════════════════
# PART 7 — HEAD-TO-HEAD vs QASM PAPER (Cell 2)
# ════════════════════════════════════════════

print("=" * 70)
print("PART 7 — HEAD-TO-HEAD: QASM PAPER vs QASM DIAGONAL")
print("=" * 70)

paper_raw_avg    = np.mean([qasm_paper_raw_oracle[n]['f1']    for n in qasm_paper_raw_oracle])
paper_otsu_o_avg = np.mean([qasm_paper_otsu_oracle[n]['f1']   for n in qasm_paper_otsu_oracle])
paper_otsu_q_avg = np.mean([qasm_paper_otsu_quantum[n]['f1']  for n in qasm_paper_otsu_quantum])

comparisons = [
    ("Raw Oracle F1",    paper_raw_avg,    np.mean(raw_f1)),
    ("Otsu Oracle F1",   paper_otsu_o_avg, np.mean(otsu_ora_f1)),
    ("Otsu Quantum F1",  paper_otsu_q_avg, np.mean(otsu_qnt_f1)),
]

print(
    f"\n{'Metric':<25}"
    f"{'QASM Paper':<14}"
    f"{'QASM Diagonal':<14}"
    f"{'Δ':<10}"
)
print("-" * 65)

for label, paper_val, diag_val in comparisons:
    delta  = diag_val - paper_val
    symbol = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(
        f"  {label:<23}"
        f"{paper_val:<14.4f}"
        f"{diag_val:<14.4f}"
        f"{symbol}{abs(delta):.4f}"
    )

print("-" * 65)
print()

# per-image table
print(
    f"{'Image':<15}"
    f"{'Paper Otsu-Q F1':<18}"
    f"{'Diag Otsu-Q F1':<18}"
    f"{'Δ':<10}"
)
print("-" * 70)

for name in qasm_diag_results:
    pf = qasm_paper_otsu_quantum[name]['f1']
    df = qasm_diag_otsu_quantum[name]['f1']
    delta  = df - pf
    symbol = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(
        f"{name:<15}"
        f"{pf:<18.4f}"
        f"{df:<18.4f}"
        f"{symbol}{abs(delta):.4f}"
    )

print("-" * 70)
print()
print("Stored in:")
print("  qasm_diag_results       — prob_map, grad, edge_mask per image")
print("  qasm_diag_raw_oracle    — Raw oracle eval")
print("  qasm_diag_otsu_oracle   — Otsu Oracle F1")
print("  qasm_diag_otsu_quantum  — Otsu Quantum F1")